# Crypto Transaction Schema Analysis v2

Actual downloaded data is separated from synthetic policy fields. This notebook does not modify BE contracts or runtime APIs.

## Context & Inputs

Load the actual public Bitcoin OTC graph file, regulatory required-field mapping, synthetic v2 dataset, and policy sensitivity results.

In [ ]:
# ruff: noqa: E501, E701, E702, I001

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name != 'ADP-DA': ROOT = next(p for p in [ROOT, *ROOT.parents] if p.name == 'ADP-DA')
actual = pd.read_csv(ROOT/'03_digital_asset/data/raw/transactions/soc-sign-bitcoinotc.csv')
inventory = pd.read_csv(ROOT/'03_digital_asset/data/raw/crypto_card/public_transaction_dataset_inventory.csv')
mapping = pd.read_csv(ROOT/'03_digital_asset/data/processed/regulatory_transaction_field_mapping.csv')
synthetic = pd.read_csv(ROOT/'03_digital_asset/data/processed/synthetic_regulated_transfer_v2.csv')
sensitivity = pd.DataFrame(json.loads((ROOT/'03_digital_asset/artifacts/candidate_policy_v2/policy_sensitivity_results.json').read_text(encoding='utf-8')))
profile = json.loads((ROOT/'03_digital_asset/data/processed/actual_transaction_profile.json').read_text(encoding='utf-8'))
print({'actual_rows': len(actual), 'synthetic_rows': len(synthetic), 'required_fields': len(mapping)})

## 1. Row Count, Columns, Types, Missingness

This verifies what execution-like fields actually exist in the downloaded file.

In [ ]:
display(actual.head())
display(pd.DataFrame({'column': actual.columns, 'dtype': [str(t) for t in actual.dtypes], 'missing_rate': actual.isna().mean().round(4).values}))

## 2. Identifier, Address, Status, Graph, Label Availability

The downloaded file has an event id and graph edges, but no blockchain transaction hash, address, amount, or execution status.

In [ ]:
availability = {k: profile[k] for k in ['duplicate_transaction','tx_identifier_uniqueness','address_availability','amount_distribution','status_confirmation_field_exists','graph_link_info_exists','label_risk_info_exists']}
print(availability)

## 3. Actual Temporal Distribution And Frequency

Frequency is measured from actual event timestamps.

In [ ]:
actual['event_day'] = pd.to_datetime(actual['event_time']).dt.date
freq = actual.groupby('event_day').size().reset_index(name='events')
display(freq.describe())
freq.plot(x='event_day', y='events', legend=False, color='#3f7f93', title='Actual Bitcoin OTC Event Frequency')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

## 4. Sender/Receiver Concentration

Source and target user concentration are actual graph properties and are used for synthetic v2 concentration only.

In [ ]:
sender_top = actual['source_user_id'].value_counts().head(10).reset_index(); sender_top.columns=['source_user_id','count']
receiver_top = actual['target_user_id'].value_counts().head(10).reset_index(); receiver_top.columns=['target_user_id','count']
display(sender_top); display(receiver_top)
sender_top.plot.bar(x='source_user_id', y='count', legend=False, color='#446fb3', title='Top Actual Senders')
plt.tight_layout()

## 5. Amount Distribution

The actual downloaded file has no amount. The v2 amount column is therefore explicitly synthetic and must not be interpreted as observed market volume.

In [ ]:
synthetic['requested_amount'].plot.hist(bins=30, color='#4f8f65', title='Synthetic v2 Requested Amount Distribution')
plt.tight_layout()
display(synthetic['requested_amount'].describe())

## 6. Regulatory -> Transaction Field Mapping

Map each regulatory required field to an actual dataset column and availability class.

In [ ]:
display(mapping[['required_field','legal_control','actual_dataset_field','availability_class','confidence']])

## 7. Regulatory Field Availability Distribution

In [ ]:
availability = mapping['availability_class'].value_counts().rename_axis('availability_class').reset_index(name='count')
display(availability)
availability.plot.bar(x='availability_class', y='count', legend=False, color='#3f7f93', title='Regulatory Field Availability')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

## 8. Control-Level Coverage

Control readiness is quantified from mapped required fields, not inferred from legal text alone.

In [ ]:
gap = json.loads((ROOT/'03_digital_asset/artifacts/candidate_policy_v2/schema_gap_detailed.json').read_text(encoding='utf-8'))
print(gap['metrics'])

## 9. Policy Removal Sensitivity

Compare baseline against removal scenarios for PASS/BLOCK/REVIEW and false allow/block deltas.

In [ ]:
display(sensitivity)
sensitivity.set_index('scenario')[['pass_count','block_count','review_count']].plot.bar(color=['#3f7f93','#446fb3','#9a6b45'], title='Policy Removal Decision Counts')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

## 10. False Allow Sensitivity

In [ ]:
sensitivity.set_index('scenario')['false_allow'].plot.bar(color='#b35a4a', title='False Allow by Policy Removal')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

## 11. Statistical Support

Use effect sizes where the data supports it. Cramer's V is calculated for categorical associations; no unsupported p-value-only claims are made.

In [ ]:
stats = json.loads((ROOT/'03_digital_asset/data/processed/statistical_support_v2.json').read_text(encoding='utf-8'))
print(stats)

## Takeaways

Supported: H1, H2, H3, H4. H5 remains a design requirement because the actual downloaded file lacks transaction hash and execution status fields.